# ShipBrain - train the model (free Colab T4 GPU is enough)

Runtime -> Change runtime type -> **T4 GPU**. Then run the cells from top to bottom (about 1-3 hours).

What you need:
* nothing else for the public tasks (meshing, FE tools, CAD, checks);
* optional, private: your `rules_index.json` (rule-book retrieval tasks). Put it in your Google Drive, never in the repo.
  If you train with it, keep the trained model **private** (a model can repeat what it was trained on).
* to save the model on Hugging Face: a **write** token stored in Colab's *Secrets* panel (key icon on the left) under the name
  `HF_TOKEN`. Never paste a token into a cell or a chat.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. code (public repo) + training libraries
!git clone -b claude/new-session-qpeo6g --depth 1 https://github.com/shashank-rakshe/hyperx-python.git
%cd hyperx-python/shipbrain
!pip -q install unsloth numpy scipy pyNastran ezdxf pdfplumber pypdfium2 pillow
# CadQuery adds the CAD tasks to the data (optional, ~3 min to install):
!pip -q install cadquery || echo "cadquery not installed - CAD tasks are skipped" 

In [ ]:
# 2. (optional) private rule index from Google Drive - stays in this Colab session
USE_RULES = False   # set True if you put rules_index.json in MyDrive/shipbrain/
RULES = '/content/drive/MyDrive/shipbrain/rules_index.json'
if USE_RULES:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# 3. training data: every number and file in it is produced by the real tools
N = 4000          # 4000 samples ~ 1.5 h of training on a T4 for a 4B model; use 1500 for a quick first run
extra = f'--knowledge {RULES}' if USE_RULES else ''
!python training/make_dataset.py --n {N} --eval-n 200 --out data/train.jsonl --eval-out data/eval.jsonl {extra}

In [ ]:
# 4. score the base model BEFORE training (tool choice + exact arguments)
BASE = 'unsloth/Qwen3-1.7B'     # fast on a CPU; unsloth/Qwen3-4B-Instruct-2507 = better answers but ~2x slower
!python training/evaluate.py --local {BASE} --data data/eval.jsonl --limit 60 --out eval_before.json

In [ ]:
# 5. fine-tune (QLoRA) and export the CPU model (GGUF)
!python training/finetune.py --data data/train.jsonl --base {BASE} --out shipbrain_model --quant q4_k_m
!ls -lh shipbrain_model

In [ ]:
# 6. score AFTER training - compare with step 4
!python training/evaluate.py --local shipbrain_model/lora --data data/eval.jsonl --limit 60 --out eval_after.json
import json
for f in ('eval_before.json', 'eval_after.json'):
    t = json.load(open(f))['total']; print(f, t)

In [ ]:
# 7. save the model to YOUR Hugging Face account (private repo) - token from Colab Secrets (HF_TOKEN)
from google.colab import userdata
from huggingface_hub import HfApi, create_repo
import glob, os
token = userdata.get('HF_TOKEN')
api = HfApi(token=token)
user = api.whoami()['name']
repo = f'{user}/shipbrain-gguf'
create_repo(repo, private=True, exist_ok=True, token=token)
for f in glob.glob('shipbrain_model/*.gguf') + ['eval_before.json', 'eval_after.json']:
    api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=repo)
print('uploaded to https://huggingface.co/' + repo)

## On the workstation (CPU, no GPU)
1. Download `shipbrain-q4_k_m.gguf` from your Hugging Face repo (or from Colab's file panel).
2. `pip install -r requirements.txt` in `shipbrain/`, then
   `python -m shipbrain --model shipbrain-q4_k_m.gguf --allow-code --knowledge D:\\ship_src --plugin D:\\cad-llm-repo`
3. Or one file: see `packaging/make_llamafile.md` and `packaging/build_exe.py`.